In [ ]:
import logging

from library.circuitry import Circuitry
from library.junction_patch import JunctionPatch
from library.qubit_array import QubitArray
from library.surface_code.expanding_patch import ExpandingSurfaceCodePatch
from library.surface_code.patch import SurfaceCodePatch
from library.common import Pauli
from library.steane_code.patch import SteaneCodePatch
from utils.simulation.clifft import sample

logging.basicConfig(level=logging.ERROR)

In [ ]:
TARGET_DISTANCE = 9

SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 9:
    raise ValueError("TARGET_DISTANCE must be odd and above 9.")

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
observables = [Pauli.X, Pauli.Z]
scenarios: dict[str, Circuitry] = dict()
point = 0

In [ ]:
# Generate circuit up to and including preparation with T-injection
for observable in observables:
    qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(
        qubits,
        anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
        injection=SteaneCodePatch.Injection.T,
    )

    # Append preparation
    steane.append_preparation(circuitry)
    circuitry.append_observable(
        0, f"{observable}_OBSERVABLE_PREPARED", steane.logical(observable)
    )

    scenarios[rf"Prepared [$\overline{{\mathbf{{{observable.name}}}}}$]"] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.prepared.{observable.name.lower()}-basis"
    )
point += 1

In [ ]:
# Generate circuit up to and including superdense code cycles
for observable in observables:
    qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(
        qubits,
        anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
        injection=SteaneCodePatch.Injection.T,
    )

    steane.append_preparation(circuitry)
    for s in range(SUPERDENSE_ROUNDS):
        label = f"SDC{s}"
        steane.append_superdense_cycle(circuitry, prefix=label)

    steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)
    circuitry.append_observable(
        0, f"{observable}_OBSERVABLE_SUPERDENSED", steane.logical(observable)
    )

    scenarios[
        rf"SDCx{SUPERDENSE_ROUNDS} [$\overline{{\mathbf{{{observable.name}}}}}$]"
    ] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.superdense.{observable.name.lower()}-basis"
    )
point += 1

In [ ]:
# Generate circuit up to and including double-check-T
for observable in observables:
    qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(
        qubits,
        anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
        injection=SteaneCodePatch.Injection.T,
    )

    steane.append_preparation(circuitry)
    for s in range(SUPERDENSE_ROUNDS):
        label = f"SDC{s}"
        steane.append_superdense_cycle(circuitry, prefix=label)
    steane.append_cultivation(circuitry, prefix="CULT")

    steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS)
    circuitry.append_observable(
        0, f"{observable}_OBSERVABLE_DOUBLE_CHECKED", steane.logical(observable)
    )

    scenario = (
        rf"Double-Check-$\mathbf{{T}}$ [$\overline{{\mathbf{{{observable.name}}}}}$]"
    )
    scenarios[scenario] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.double-check-t.{observable.name.lower()}-basis"
    )
point += 1

In [ ]:
def inactive_source(location: tuple[float, float]) -> bool:
    return location[1] == TARGET_DISTANCE - 4.5


# Generate circuit up to and including teleportation
for observable in observables:
    qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(
        qubits,
        anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
        injection=SteaneCodePatch.Injection.T,
    )
    junction = JunctionPatch(qubits, anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 5))
    source = SurfaceCodePatch(
        qubits, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4)
    )

    steane.append_preparation(circuitry)
    for s in range(SUPERDENSE_ROUNDS):
        label = f"SDC{s}"
        steane.append_superdense_cycle(circuitry, prefix=label)
    steane.append_cultivation(circuitry, prefix="CULT")
    for rnd in range(TELEPORT_ROUNDS):
        for mmt in steane.TELEPORTATION_MOMENTS:
            steane.append_teleportation(circuitry, moment=mmt, prefix=f"TPT{rnd}")
            junction.append_syndrome(circuitry, moment=mmt, prefix=f"JCT{rnd}")
            source.append_round(
                circuitry,
                moment=mmt,
                prepare=Pauli.X if rnd == 0 else None,
                prefix=f"SC{rnd}",
                inactive=inactive_source,
            )
            circuitry.append_tick()

    for mmt in steane.DESTRUCTION_MOMENTS:
        steane.append_destruction(circuitry, moment=mmt)
        source.append_round(circuitry, moment=mmt, prefix=f"SC{TELEPORT_ROUNDS}")
        circuitry.append_tick()

    match observable:
        case Pauli.X:
            extras = ["TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6"]
        case Pauli.Y:
            extras = [
                "JCT0:Z0",
                "JCT0:Z1",
                "JCT0:Z2",
                "TPT0:XB",
                "TPT1:XB",
                "TPT2:XB",
                "DST:X1",
                "DST:X5",
                "DST:X6",
            ]
        case Pauli.Z:
            extras = ["JCT0:Z0", "JCT0:Z1", "JCT0:Z2"]
    circuitry.append_observable(
        0,
        f"{observable}_OBSERVABLE_TELEPORTED",
        source.logical(observable),
        *extras,
        flip=observable == Pauli.Y,
    )

    steane.annotate_detectors(
        circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS
    )
    junction.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS)
    source.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS + 1, prepared=Pauli.X)

    scenario = rf"Teleported [{'-' if observable == Pauli.Y else ''}$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.teleported.{observable.name.lower()}-basis"
    )
point += 1

In [ ]:
# Generate the circuit up to and including expansion :)
for observable in observables:
    qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(
        qubits,
        anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
        injection=SteaneCodePatch.Injection.T,
    )
    junction = JunctionPatch(qubits, anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 5))
    source = SurfaceCodePatch(
        qubits, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4)
    )
    expanding = ExpandingSurfaceCodePatch(
        qubits, distance=5, anchor=(1, 1), expansion=TARGET_DISTANCE - 5
    )

    steane.append_preparation(circuitry)
    for s in range(SUPERDENSE_ROUNDS):
        label = f"SDC{s}"
        steane.append_superdense_cycle(circuitry, prefix=label)
    steane.append_cultivation(circuitry, prefix="CULT")
    for rnd in range(TELEPORT_ROUNDS):
        for mmt in steane.TELEPORTATION_MOMENTS:
            steane.append_teleportation(circuitry, moment=mmt, prefix=f"TPT{rnd}")
            junction.append_syndrome(circuitry, moment=mmt, prefix=f"JCT{rnd}")
            source.append_round(
                circuitry,
                moment=mmt,
                prepare=Pauli.X if rnd == 0 else None,
                prefix=f"SC{rnd}",
                inactive=inactive_source,
            )
            circuitry.append_tick()

    for mmt in steane.DESTRUCTION_MOMENTS:
        steane.append_destruction(circuitry, moment=mmt)
        source.append_round(circuitry, moment=mmt, prefix=f"SC{TELEPORT_ROUNDS}")
        circuitry.append_tick()

    for mmt in expanding.MOMENTS:
        expanding.append_expansion(circuitry, moment=mmt, prefix="EXP")
        circuitry.append_tick()

    match observable:
        case Pauli.X:
            extras = ["TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6"]
        case Pauli.Y:
            extras = [
                "JCT0:Z0",
                "JCT0:Z1",
                "JCT0:Z2",
                "TPT0:XB",
                "TPT1:XB",
                "TPT2:XB",
                "DST:X1",
                "DST:X5",
                "DST:X6",
            ]
        case Pauli.Z:
            extras = ["JCT0:Z0", "JCT0:Z1", "JCT0:Z2"]
    circuitry.append_observable(
        0,
        f"{observable}_OBSERVABLE_EXPANDED",
        expanding.logical(observable),
        *extras,
        flip=observable == Pauli.Y,
    )

    steane.annotate_detectors(
        circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS
    )
    junction.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS)
    source.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS + 1, prepared=Pauli.X)
    expanding.annotate_detectors(
        circuitry, sc_rounds=TELEPORT_ROUNDS + 1, source=source
    )

    scenario = rf"Expanded [{'-' if observable == Pauli.Y else ''}$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.expanded.{observable.name.lower()}-basis"
    )
point += 1

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{T}\rangle$ [Corrected $\frac{\overline{\mathbf{X}}+\overline{\mathbf{Z}}}{\sqrt{2}}$]"
sample(
    scenarios,
    title=title,
    label="Point",
    shots=1e6,
    correction=True,
    figsize=(16, 5),
    fontsize=8,
)

In [ ]:
# simulate(scenarios, title, label="Point", postselection=True, shots=1e3, minimal_noise=-3, figsize=(11, 4.5), num_workers=7)